# SI stress test — support fraction and finite-climate-panel uncertainty

Purpose: test whether the existing **single–tensor climate-ordering comparison** is sensitive to excluding low-support climates, and quantify finite-panel uncertainty in Kendall $\tau$ by **paired climate bootstrap**.

This notebook is deliberately restricted to SI robustness. It does **not** choose a minimum coverage criterion, introduce $H/N$, classify material/climate validity, interpret process performance, or treat temporal modes as independent replicates. The cutoff grid is a sensitivity grid only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
from scipy.stats import kendalltau

ROOT = Path('/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone')
SCORES = ROOT / 'recent_experimental_results/manuscript_required_outputs_only/results/P0_submission_gate_outputs_2026-10-14/P0_tensor_outputs/P0_tensor_climate_scores.csv'
WEATHER = ROOT / 'process_bridge/inputs_common/climate_hourly_2015_2024.csv'
OUT = ROOT / 'recent_experimental_results/manuscript_required_outputs_only/results/support_fraction_rank_uncertainty_2026-10-27'

# Sensitivity grid only; these are not proposed validity thresholds.
CUTOFFS_PCT = [0, 1, 2.5, 5, 10, 20]
SI_CUTOFFS_PCT = [0, 2.5, 5, 10]  # 0 = unfiltered in final SI table
N_BOOT = 10000
SEED = 20261027

assert SCORES.is_file(), f'Missing exact input: {SCORES}'
assert WEATHER.is_file(), f'Missing exact input: {WEATHER}'
OUT.mkdir(parents=True, exist_ok=True)
print('SCORES :', SCORES)
print('WEATHER:', WEATHER)
print('OUT    :', OUT)

## 1. Load and audit schema

No path discovery or fallback is used. Column resolution below is limited to semantic aliases because the upstream CSV schema may use different labels. Ambiguous or absent required fields cause a hard failure rather than an inferred analysis.

In [ ]:
scores_raw = pd.read_csv(SCORES)
weather_raw = pd.read_csv(WEATHER)
print('scores shape :', scores_raw.shape)
print('weather shape:', weather_raw.shape)
print('\nSCORES columns:')
print(scores_raw.columns.tolist())
print('\nWEATHER columns:')
print(weather_raw.columns.tolist())
print('\nSCORES preview:')
display(scores_raw.head())

In [ ]:
def resolve_col(df, aliases, label, required=True):
    hits = [x for x in aliases if x in df.columns]
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        raise ValueError(f'Ambiguous {label}: found {hits}. Set the mapping explicitly.')
    if required:
        raise KeyError(f'Missing {label}. Expected exactly one of {aliases}. Available: {df.columns.tolist()}')
    return None

# Explicit semantic mapping. Extend aliases only after inspecting the printed upstream schema.
COL = {}
COL['city'] = resolve_col(scores_raw, ['city','City'], 'city')
COL['material'] = resolve_col(scores_raw, ['material','Material','sorbent'], 'material')
COL['branch'] = resolve_col(scores_raw, ['branch'], 'branch')
COL['mode'] = resolve_col(scores_raw, ['temporal_mode','mode','time_mode'], 'temporal mode', required=False)
COL['single'] = resolve_col(scores_raw, ['H_single','H_iso','single_score','score_single','H_single_isotherm'], 'single-isotherm score')
COL['tensor'] = resolve_col(scores_raw, ['H_tensor','H_RHT','tensor_score','score_tensor'], 'tensor score')

# Preferred: upstream support fraction already attached to each climate score.
COL['support_fraction'] = resolve_col(scores_raw, ['support_fraction','common_support_fraction','supported_fraction','coverage_fraction'], 'support fraction', required=False)
COL['n_supported'] = resolve_col(scores_raw, ['n_supported','n_valid','n_supported_transitions','valid_transitions','n_common_transitions'], 'supported-transition count', required=False)

WCITY = resolve_col(weather_raw, ['city','City'], 'weather city')
WTIME = resolve_col(weather_raw, ['timestamp_utc','timestamp','datetime','time'], 'weather timestamp')

print(json.dumps(COL, indent=2))
print('weather:', {'city': WCITY, 'time': WTIME})

## 2. Construct the climate-level support fraction

The retained quantity is the fraction of possible transitions in the corresponding **city × temporal-mode representation** that are represented by the exact common-support score. The denominator is therefore $N_{possible}(city, temporal\ mode)$, reconstructed from the weather timestamps using the same nominal temporal representations (subsample 1/3/6/24 h; average 6/24 h). No equilibrium extrapolation or new coverage criterion is introduced.


In [ ]:
scores = scores_raw.copy()
weather = weather_raw[[WCITY, WTIME]].copy()
weather[WTIME] = pd.to_datetime(weather[WTIME], errors='coerce', utc=True)
if weather[WTIME].isna().any():
    raise ValueError('WEATHER contains unparsable timestamps; support denominator is not safe to construct.')

# N_possible must be specific to city × temporal_mode.
# Construct only the six canonical temporal representations used by P0.
CANONICAL_MODES = ['subsample_1h','subsample_3h','subsample_6h','subsample_24h','average_6h','average_24h']
observed_modes = sorted(scores[COL['mode']].dropna().astype(str).unique())
unknown_modes = sorted(set(observed_modes) - set(CANONICAL_MODES))
if unknown_modes:
    raise ValueError(f'Unexpected temporal_mode values: {unknown_modes}. Denominator construction is intentionally restricted to canonical P0 modes.')

def n_possible_by_mode(ts):
    ts = pd.Series(pd.DatetimeIndex(ts).drop_duplicates().sort_values())
    out = {}
    # Subsample representations: retain every k-th record from the ordered hourly trajectory.
    for k in (1,3,6,24):
        nr = len(ts.iloc[::k])
        out[f'subsample_{k}h'] = max(nr - 1, 0)
    # Average representations: one state per occupied UTC-aligned k-hour bin.
    # Only timestamps are needed to count possible transitions; H itself is not recomputed here.
    idx = pd.DatetimeIndex(ts)
    for k in (6,24):
        nr = idx.floor(f'{k}h').nunique()
        out[f'average_{k}h'] = max(int(nr) - 1, 0)
    return out

possible_rows=[]
for city, g in weather.groupby(WCITY, sort=True):
    counts=n_possible_by_mode(g[WTIME])
    for mode, npos in counts.items():
        possible_rows.append({COL['city']:city, COL['mode']:mode, 'n_possible_transitions':npos})
n_possible = pd.DataFrame(possible_rows)

if COL['support_fraction'] is not None:
    sf = pd.to_numeric(scores[COL['support_fraction']], errors='coerce')
    if sf.dropna().empty:
        raise ValueError('Support-fraction column exists but contains no numeric values.')
    if sf.max(skipna=True) > 1.0 + 1e-12:
        if sf.max(skipna=True) <= 100.0 + 1e-12:
            sf = sf / 100.0
        else:
            raise ValueError('Support fraction exceeds 100%; definition cannot be inferred safely.')
    scores['support_fraction'] = sf
elif COL['n_supported'] is not None:
    scores = scores.merge(n_possible, on=[COL['city'],COL['mode']], how='left', validate='many_to_one')
    if scores['n_possible_transitions'].isna().any():
        bad=scores.loc[scores['n_possible_transitions'].isna(), [COL['city'],COL['mode']].drop_duplicates()]
        raise ValueError(f'Missing city × temporal-mode denominator:\n{bad}')
    scores['support_fraction'] = pd.to_numeric(scores[COL['n_supported']], errors='coerce') / scores['n_possible_transitions']
else:
    raise KeyError('Cannot construct support_fraction: SCORES must contain either an explicit support-fraction column or a supported-transition count. No proxy will be invented.')

if ((scores['support_fraction'] < -1e-12) | (scores['support_fraction'] > 1+1e-12)).any():
    bad = scores.loc[(scores['support_fraction'] < -1e-12) | (scores['support_fraction'] > 1+1e-12), [COL['city'],COL['mode'],'support_fraction']].head(20)
    raise ValueError(f'Support fraction outside [0,1]. Inspect upstream definition.\n{bad}')

scores['support_fraction'] = scores['support_fraction'].clip(0,1)
scores['support_pct'] = 100*scores['support_fraction']
scores['H_single'] = pd.to_numeric(scores[COL['single']], errors='coerce')
scores['H_tensor'] = pd.to_numeric(scores[COL['tensor']], errors='coerce')

# Each analysis row is a climate within a fixed material × branch × temporal-mode cell.
cell_cols = [COL['material'], COL['branch'], COL['mode']]
key_cols = cell_cols + [COL['city']]
if scores.duplicated(key_cols).any():
    dup = scores.loc[scores.duplicated(key_cols, keep=False), key_cols].sort_values(key_cols).head(30)
    raise ValueError(f'Expected one row per material × branch × temporal_mode × climate; duplicates found.\n{dup}')

score_out_cols = key_cols + ['H_single','H_tensor','support_fraction','support_pct']
if 'n_possible_transitions' in scores.columns:
    score_out_cols += [COL['n_supported'], 'n_possible_transitions']
support_scores = scores[score_out_cols].copy()
support_scores.to_csv(OUT/'support_fraction_climate_scores.csv', index=False)
display(support_scores.head())


## 3. Cutoff sensitivity of single–tensor Kendall $\tau$

For cutoff $c>0$, a climate is retained when its common-support fraction is at least $c$%. `0%` is explicitly the unfiltered panel. Each material × branch × temporal-mode cell is analyzed separately; the six temporal modes are never pooled or counted as replicate observations.


In [ ]:
def tau_safe(x, y):
    z = pd.DataFrame({'x':x,'y':y}).dropna()
    if len(z) < 3 or z['x'].nunique() < 2 or z['y'].nunique() < 2:
        return np.nan
    return float(kendalltau(z['x'], z['y'], variant='b', nan_policy='omit').statistic)

rows=[]
for keys, g in support_scores.groupby(cell_cols, dropna=False, sort=True):
    if not isinstance(keys, tuple): keys=(keys,)
    meta=dict(zip(cell_cols, keys))
    base=g.dropna(subset=['H_single','H_tensor']).copy()
    tau0=tau_safe(base.H_single, base.H_tensor)
    for cutoff in CUTOFFS_PCT:
        kept = base if cutoff == 0 else base.loc[base.support_pct >= cutoff]
        tau=tau_safe(kept.H_single, kept.H_tensor)
        rows.append({**meta,'cutoff_pct':cutoff,'n_climates':len(kept),'n_removed':len(base)-len(kept),
                     'tau':tau,'tau_unfiltered':tau0,'delta_tau_vs_unfiltered':tau-tau0 if np.isfinite(tau) and np.isfinite(tau0) else np.nan,
                     'min_support_pct_retained':kept.support_pct.min() if len(kept) else np.nan})

panel = pd.DataFrame(rows)
display(panel)

## 4. Paired climate bootstrap

Bootstrap unit = climate. For each retained panel, climates are resampled with replacement and the corresponding single and tensor scores are carried together as a pair. Each material × branch × temporal-mode × cutoff cell receives a deterministic seed derived from its key, so results do not depend on cell ordering. Only bootstrap summaries are retained; raw draws are not written.


In [ ]:
import hashlib

def deterministic_seed(meta, cutoff):
    key = '|'.join([str(SEED)] + [str(meta[c]) for c in cell_cols] + [str(cutoff)])
    digest = hashlib.sha256(key.encode('utf-8')).digest()
    return int.from_bytes(digest[:8], 'little', signed=False)

summary_rows=[]
for keys, g in support_scores.groupby(cell_cols, dropna=False, sort=True):
    if not isinstance(keys, tuple): keys=(keys,)
    meta=dict(zip(cell_cols, keys))
    base=g.dropna(subset=['H_single','H_tensor']).reset_index(drop=True)
    for cutoff in CUTOFFS_PCT:
        kept = base if cutoff == 0 else base.loc[base.support_pct >= cutoff].reset_index(drop=True)
        n=len(kept); obs=tau_safe(kept.H_single, kept.H_tensor)
        vals=np.full(N_BOOT, np.nan)
        cell_seed=deterministic_seed(meta, cutoff)
        rng=np.random.default_rng(cell_seed)
        if n >= 3:
            xy=kept[['H_single','H_tensor']].to_numpy(float)
            for b in range(N_BOOT):
                idx=rng.integers(0,n,size=n)
                vals[b]=tau_safe(xy[idx,0],xy[idx,1])
        finite=vals[np.isfinite(vals)]
        lo,med,hi=(np.quantile(finite,[.025,.5,.975]) if len(finite) else [np.nan]*3)
        summary_rows.append({**meta,'cutoff_pct':cutoff,'n_climates':n,'tau_observed':obs,
                             'bootstrap_seed':cell_seed,'bootstrap_n_requested':N_BOOT,
                             'bootstrap_n_finite':len(finite),'tau_boot_median':med,
                             'tau_boot_q025':lo,'tau_boot_q975':hi})

bsum=pd.DataFrame(summary_rows)
panel2=panel.merge(bsum,on=cell_cols+['cutoff_pct','n_climates'],how='left',validate='one_to_one')
panel2.to_csv(OUT/'support_fraction_tau_by_cell.csv', index=False)
display(panel2)


## 5. Identify sensitivity, without defining a new validity rule

`support_fraction_sensitive_cells.csv` is an audit table, not a pass/fail classification. It reports each cell's maximum absolute change in $\tau$ over the prespecified cutoff grid and the cutoff at which that maximum occurs. No threshold is imposed on what counts as “acceptable.”

In [ ]:
sens_rows=[]
for keys,g in panel2.groupby(cell_cols,dropna=False,sort=True):
    if not isinstance(keys,tuple): keys=(keys,)
    meta=dict(zip(cell_cols,keys))
    finite=g.dropna(subset=['delta_tau_vs_unfiltered']).copy()
    if len(finite):
        j=finite['delta_tau_vs_unfiltered'].abs().idxmax(); r=finite.loc[j]
        sens_rows.append({**meta,'tau_unfiltered':r['tau_unfiltered'],'max_abs_delta_tau':abs(r['delta_tau_vs_unfiltered']),
                          'cutoff_at_max_abs_delta_pct':r['cutoff_pct'],'tau_at_max_abs_delta':r['tau'],
                          'n_climates_at_max_abs_delta':r['n_climates']})
    else:
        sens_rows.append({**meta,'tau_unfiltered':np.nan,'max_abs_delta_tau':np.nan,'cutoff_at_max_abs_delta_pct':np.nan,
                          'tau_at_max_abs_delta':np.nan,'n_climates_at_max_abs_delta':np.nan})
sensitive=pd.DataFrame(sens_rows)
sensitive.to_csv(OUT/'support_fraction_sensitive_cells.csv',index=False)
display(sensitive)

## 6. Final SI table

The final SI-facing table retains only unfiltered, 2.5%, 5%, and 10% panels. The 1% and 20% panels remain in the full sensitivity outputs. No cutoff is selected as a recommended minimum.

In [ ]:
si=panel2.loc[panel2.cutoff_pct.isin(SI_CUTOFFS_PCT)].copy()
si['panel_label']=si['cutoff_pct'].map({0:'unfiltered',2.5:'support >= 2.5%',5:'support >= 5%',10:'support >= 10%'})
front=cell_cols+['panel_label','cutoff_pct','n_climates','n_removed','tau','delta_tau_vs_unfiltered','tau_boot_q025','tau_boot_median','tau_boot_q975','bootstrap_n_finite']
si=si[front+[x for x in si.columns if x not in front]]
si.to_csv(OUT/'SI_support_fraction_rank_uncertainty.csv',index=False)
display(si)

expected=[
'support_fraction_climate_scores.csv',
'support_fraction_tau_by_cell.csv',
'support_fraction_sensitive_cells.csv',
'SI_support_fraction_rank_uncertainty.csv']
print('\nWritten outputs:')
for f in expected:
    p=OUT/f
    assert p.is_file() and p.stat().st_size>0, f'Output missing/empty: {p}'
    print(f'  {f}: {p.stat().st_size:,} bytes')


### Interpretation boundary

A routine pass of this stress test supports reporting the result in SI rather than creating a new Results subsection. The analysis only asks whether climate ordering is stable to low-support-climate exclusion and how much Kendall $\tau$ varies under finite-panel resampling. It does not establish a minimum coverage criterion or convert equilibrium opportunity into process performance.